In [9]:
!pip install pyserial

In [10]:
import serial, time
!pip install pyserial

In [68]:
#ser.close()

**Note:** if importing `serial` causes an error, you need to install the `pyserial` module using `pip`:

`pip install pyserial`

or 

`pip3 install pyserial`

Windows users should use the Anaconda prompt.  Mac users should be able to use the terminal.

In [69]:
print(serial)

<module 'serial' from 'C:\\Users\\boome\\anaconda3\\Lib\\site-packages\\serial\\__init__.py'>


In [70]:
print(serial.__file__)

C:\Users\boome\anaconda3\Lib\site-packages\serial\__init__.py


In [71]:
print(serial.__version__)

3.5


In [72]:
serial.VERSION

'3.5'

**Note:** if you serial version is 2.x, we might need to make changes to the code below

In [73]:
baudrate = 115200

In [74]:
#portname = '/dev/cu.usbmodem11301'#mac
portname = 'COM4'#windows

In [75]:
ser = serial.Serial(portname, baudrate, timeout=5)

In [76]:
ser.in_waiting

0

In [77]:
def read_all(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [78]:
read_all(ser)

''

In [79]:
def read_one_line(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        if data1 in ['\n','\r']:
            break
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [80]:
read_one_line(ser)

'dual servo control over serial'

In [81]:
read_all(ser)

''

In [82]:
def one_byte_int_to_serial_byte(int_byte):
    out_byte = int(int_byte).to_bytes(1, byteorder='big')
    return out_byte

In [83]:
def WriteByte(ser, bytein):
    out_byte = one_byte_int_to_serial_byte(bytein)
    ser.write(out_byte)

# Break an integer into two bytes

In [84]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from numpy import sin, cos, tan, pi
import robotics
from robotics import Rx, Ry, Rz, sind, cosd, DH, prettymat
rtd = 180/pi
dtr = pi/180

## Parameters
![Screenshot 2026-06-19 234816.png](attachment:21bb921d-4bfa-4f88-9196-fb7a3789d1a5.png)

In [97]:
# measure based off sketch parameters
# units of cm
A = 4.5*2.54     # height of base of l1
B = 1*2.54     # offset from z1 to l1|
C = 2.5*2.54     # offset from l1 to l2
D = 0.5*2.54     # offset from l2 to center of EOAT
l1 = 24   # base link
l2 = 21.5 # tip link
l3 = 10   # gripper link

#define pick location
pickX = 1
pickY = 1
pickZ = 1

#define place location
placeX = 10
placeY = 10
placeZ = 10

#define obstical location
obstacleX = 0
obstacleY = 0

################ for reference
#T01 = DH(0,0,th1,A)
#T12 = DH(90,0,th2+90,B)
#T23 = DH(180,l1,th3,C)
#T34 = DH(0,l2,th4,D)

## Path generation

In [98]:
lift = 4  # z displacement from pick
obstacle_radius = 2 * 2.45  # 2 inches in cm
clearance = 3 # buffer distance from obstacle (cm)

#number of positions for each path
N1 = 3  # z  down into the pick
N2 = 2  # z  up above the place z
N3 = 4  # y  to below the obstacle
N4 = 10 # x  across to the place x
N5 = 10 # y  to the place y
N6 = 3  # z  down to the place
N7 = 2  # z  retract up 

z_clear = max(pickZ, placeZ) + lift                          # travel height: above place z (and above pick)
y_safe  = obstacleY - obstacle_radius - clearance            # a lane in front of (below) the obstacle

above_pick = [pickX, pickY, z_clear]                        # start: positioned above the pick
path1 = np.linspace(above_pick, [pickX, pickY, pickZ], N1)   # z  down into the pick
# ---- PAUSE: close gripper ----
path2 = np.linspace([pickX,  pickY,  pickZ],   [pickX,  pickY,  z_clear], N2)  # z  up above the place z
path3 = np.linspace([pickX,  pickY,  z_clear], [pickX,  y_safe, z_clear], N3)  # y  to below the obstacle
path4 = np.linspace([pickX,  y_safe, z_clear], [placeX, y_safe, z_clear], N4)  # x  across to the place x
path5 = np.linspace([placeX, y_safe, z_clear], [placeX, placeY, z_clear], N5)  # y  to the place y
path6 = np.linspace([placeX, placeY, z_clear], [placeX, placeY, placeZ],  N6)  # z  down to the place
# ---- PAUSE: open gripper ----
path7 = np.linspace([placeX, placeY, placeZ],  [placeX, placeY, z_clear], N7)  # z  retract up 

In [99]:
path1

array([[ 1. ,  1. , 14. ],
       [ 1. ,  1. ,  7.5],
       [ 1. ,  1. ,  1. ]])

## Get from x,y,z postions to arduino servo values

In [94]:
# takes 3d coordinant and outputs required servo angles
def inverseKinematics(X, Y, Z):

    #define position as being relative to base
    P_tip_0 = np.array([X,Y,Z,1]) 

    #solve for th1
    th1 = np.arctan2(P_tip_0[1], P_tip_0[0])*rtd

    #define transform matrix to get to link 1 co-ordinants
    T01 = DH(0,0,th1,A)
    T10 = robotics.HTinv(T01)
    #get tip position relative to link 1
    P_tip_1 = T10 @ P_tip_0

    #distance from origin1 to tip
    r_squared = P_tip_1[0]**2 + P_tip_1[2]**2
    
    #law of cos for angle between links
    alpha_temp = (r_squared-l1**2-l2**2)/(-2*l1*l2)
    #print(alpha_temp)
    #alpha = np.arccos(alpha_temp)
    sin_alpha_p = np.sqrt(1-alpha_temp**2)
    sin_alpha_n = -sin_alpha_p
    alpha = np.arctan2(sin_alpha_p, alpha_temp)
    alpha_2 = np.arctan2(sin_alpha_n, alpha_temp)
    
    #vertical angle theorem for theta 2
    theta3 = 180 - alpha*rtd
    theta3_2 = 180 - alpha_2*rtd
    
    #triangle in link1 co-ordinant system for psi
    psi = np.arctan2(l2*sind(theta3),l1+l2*cosd(theta3))
    psi_2 = np.arctan2(l2*sind(theta3_2),l1+l2*cosd(theta3_2))
    
    #angle of r to x-axis
    beta = np.arctan2(P_tip_1[2], P_tip_1[0])
    
    #difference in beta and psi is theta 1
    theta2 = (beta - psi)*rtd
    theta2_2 = (beta - psi_2)*rtd

    th2 = theta2
    th3 = theta3 

    # l3 should be perpendicular to the ground at all times 
    # may need more logic depending on the values th2 and th3 are for the generated path
    th4 = 360 - 90 - th2 - th3 

    return th1, th2, th3, th4

In [95]:
# run each path through inverseKinemnatics function to get angles for each position
path1_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path1])
path2_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path2])
path3_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path3])
path4_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path4])
path5_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path5])
path6_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path6])
path7_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path7])
path1_angles

C:\Users\boome\AppData\Local\Temp\ipykernel_28888\4208638327.py:23: RuntimeWarning: invalid value encountered in sqrt
  sin_alpha_p = np.sqrt(1-alpha_temp**2)


array([[ 26.56505118, -75.80780354, 151.54725684, 194.26054671],
       [ 26.56505118, -85.0253788 , 149.98744209, 205.03793671],
       [ 26.56505118, -93.0037048 , 147.65793333, 215.34577147]])

In [52]:
def thetaInterpolation(th1, th2, th3, th4):
    #convert angle into arduino code
    theta_min = 0     # minimum angle
    theta_max = 180   # maximum angle
    min_new = 1000    # minimum servo value
    max_new = 2000    # maximum servo value

    #linear interpolate for the first theta value
    myint = min_new + ((th1-theta_min)*(max_new-min_new))/(theta_max-theta_min)

    #linear interpolate for the second theta value; 180 and 90 included to account for robots home position as The servo has 1000 -> theta2=90, 2000 -> theta2 = -90
    myint2 = min_new + (((180-(90+th2))-theta_min)*(max_new-min_new))/(theta_max-theta_min)

    #third theta (need to adjust once tested)
    myint3 = min_new + ((th3-theta_min)*(max_new-min_new))/(theta_max-theta_min)

    #fourth theta (need to adjust once tested)
    myint4 = min_new + ((th4-theta_min)*(max_new-min_new))/(theta_max-theta_min)

    return myint, myint2, myint3, myint4

In [53]:
# run each path_angles through thetaInterpolation function to get angles for each position in servo language
path1_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path1_angles])
path2_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path2_angles])
path3_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path3_angles])
path4_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path4_angles])
path5_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path5_angles])
path6_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path6_angles])
path7_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path7_angles])

path1_servo

array([[1062.83295819, 1762.349956  , 1853.94180715, 1908.40814886],
       [1062.83295819, 1853.26184344, 1860.33975948, 1992.92208396],
       [1062.83295819, 1944.48909706, 1853.94180715, 2090.54728991]])

In [54]:
def break_into_two(breakint):
    MSB = breakint // 256
    LSB = breakint % 256
    return MSB, LSB

In [65]:
#define arrays to hold the four bytes
byte1 = np.zeros(len(path1), dtype=int)
byte2 = np.zeros(len(path1), dtype=int)
byte3 = np.zeros(len(path1), dtype=int)
byte4 = np.zeros(len(path1), dtype=int)
byte5 = np.zeros(len(path1), dtype=int)
byte6 = np.zeros(len(path1), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path1)):
    byte1[i], byte2[i] = break_into_two(path1_servo[i,0])
    byte3[i], byte4[i] = break_into_two(path1_servo[i,1])
    byte5[i], byte6[i] = break_into_two(path1_servo[i,2])

print(byte1,'\n\n',byte2,'\n\n\n',byte3,'\n\n',byte4,'\n\n\n',byte5,'\n\n',byte6)

[4 4 4] 

 [38 38 38] 


 [6 7 7] 

 [226  61 152] 


 [7 7 7] 

 [61 68 61]


In [67]:
# Send all path points to both servos
x = 1
for i in range(len(path1)):
    WriteByte(ser, int(byte1[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6[i]))   # servo 3 LSB
    time.sleep(0.05)

    # read back confirmation from Arduino
    #line1 = read_one_line(ser)  # servo 1 bytes echo
   # line2 = read_one_line(ser)  # servo 1 int echo
    #line3 = read_one_line(ser)  # servo 2 bytes echo
    #line4 = read_one_line(ser)  # servo 2 int echo
    #print(f"Step {i}: servo1={line2}  servo2={line4}")

   # print('\ntheta1:',theta1[i],'\ntheta2:',theta2[i],'\n','\n\n')
 
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            print(response)
            if response == "Ready":
                break
            else: 
                line1 = read_one_line(ser)  # servo 1 bytes echo
                line2 = read_one_line(ser)  # servo 1 int echo
                print(f"Step {i-1}: servo1={line1}  servo2={line2}")
                line3 = read_one_line(ser)  # servo 1 bytes echo
                line4 = read_one_line(ser)  # servo 1 int echo
                print(f"Step {i}: servo1={line3}  servo2={line4}") 
                line5 = read_one_line(ser)  # servo 1 bytes echo
                line6 = read_one_line(ser)  # servo 1 int echo
                print(f"Step {i-1}: servo1={line1}  servo2={line2}")
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move


Step 0: servo1=My int: 1062  servo2=My int2: 1853
Step 1: servo1=My int3: 1860  servo2=Ready
Step 0: servo1=My int: 1062  servo2=My int2: 1853
My int: 1062
Step 0: servo1=My int2: 1944  servo2=My int3: 1853
Step 1: servo1=  servo2=
Step 0: servo1=My int2: 1944  servo2=My int3: 1853
Ready
